# ЛР2 — практика A: Titanic data pipeline

**Курс:** Нейронные сети  
**Статус:** первая самостоятельная часть ЛР2; защита общая после практики B  
**Основные библиотеки:** pandas, NumPy, matplotlib, scikit-learn  
**Датасет:** Titanic (`titanic.csv` из материалов курса)  
**Оборудование:** CPU  
**Интернет:** не требуется

> **Центральный вопрос:**  
> **Как подготовка данных влияет на корректность эксперимента?**

В этой работе один и тот же датасет проходит полный путь:

**сырая таблица → первичный анализ → фиксированный split → preprocessing → baseline → метрики → ошибки → данные для ЛР3.**

Мы не пытаемся изучить все возможности pandas и scikit-learn. Каждый инструмент появляется только тогда, когда он нужен для построения воспроизводимого ML-эксперимента.

## Паспорт работы

| Параметр | Значение |
|---|---|
| Тема | Подготовка табличных данных и классический baseline |
| Максимальный балл | 10 |
| Ожидаемое время | 1.5–2 часа |
| Reference runtime | несколько секунд на обычном CPU |
| Датасет | Titanic, 891 объект |
| Целевая переменная | `Survived` |
| Тип задачи | бинарная классификация |
| Seed | `42` |
| Результат | заполненный notebook + подготовленные массивы для ЛР3 |
| Отдельный Word-отчёт | не требуется |

### После работы вы должны уметь

- быстро исследовать табличный датасет и находить проблемы в данных;
- различать числовые и категориальные признаки;
- корректно делать `train/validation/test` split;
- объяснять, что такое **data leakage**;
- строить воспроизводимый preprocessing через `Pipeline` и `ColumnTransformer`;
- использовать простой baseline и выбирать осмысленные метрики;
- читать confusion matrix и смотреть конкретные ошибки модели;
- подготовить тот же набор данных для последующего сравнения моделей.

## Правила выполнения

- Можно пользоваться документацией, конспектами, примерами библиотек и ИИ-инструментами.
- Вы отвечаете за корректность и понимание всего кода, который сдаёте.
- Ячейки с `TODO` нужно дополнить самостоятельно.
- Не меняйте `SEED`, обязательный набор признаков и стратегию split: результат этой работы будет использован в ЛР3.
- **Test set не используется для выбора решений.** Мы посмотрим на него только после того, как pipeline и baseline будут зафиксированы.
- Существенные внешние источники и использованные ИИ-инструменты перечислите в конце notebook.

# 0. Окружение и воспроизводимость

Для лабораторной достаточно CPU. Датасет должен лежать либо рядом с notebook как `titanic.csv`,
либо в каталоге `data/titanic.csv`.

Запустите этот раздел первым.

In [ ]:
SEED = 42

from pathlib import Path
import os
import sys
import random
import platform
import importlib.metadata as md

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for package in ["numpy", "pandas", "matplotlib", "scikit-learn"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: not found")

# 1. Цель и центральный вопрос

В нейросетевых задачах легко сосредоточиться на архитектуре и забыть, что качество эксперимента начинается **до модели**.

Если preprocessing использует информацию из validation/test, если split каждый раз меняется или если категориальные признаки кодируются непоследовательно, красивое значение метрики перестаёт быть честным сравнением.

> **Центральный вопрос:**  
> **Как подготовка данных влияет на корректность эксперимента?**

К концу работы мы должны получить не только baseline, но и числовые массивы `X_train`, `X_val`, `X_test`,
которые без изменения перейдут в ЛР3.

# 2. Один датасет вместо набора несвязанных упражнений

Мы будем работать с локальным `titanic.csv`.

Каждая строка соответствует пассажиру. Цель `Survived` принимает значения:

- `0` — не выжил;
- `1` — выжил.

Сначала загрузим таблицу и проверим, что это именно ожидаемая версия датасета.

In [ ]:
candidate_paths = [
    Path("titanic.csv"),
    Path("../data/titanic.csv"),  # student notebook
    Path("data/titanic.csv"),
    Path("../../../shared/02_tabular_data/data/titanic.csv"),  # teacher master
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Файл titanic.csv не найден. Положите его рядом с notebook "
        "или в каталог data/titanic.csv."
    )

df = pd.read_csv(DATA_PATH)

expected_columns = {
    "PassengerId", "Survived", "Pclass", "Name", "Sex", "Age",
    "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"
}
assert expected_columns.issubset(df.columns), "Структура titanic.csv отличается от ожидаемой."

print("Файл:", DATA_PATH.resolve())
print("Размер таблицы:", df.shape)
display(df.head())

## 2.1. Задание 1 — первичная диагностика данных

Не нужно делать полный каталог pandas-операций. Получите только информацию, которая нужна перед построением pipeline.

1. Выведите размер таблицы и типы столбцов.
2. Посчитайте количество пропусков в каждом столбце.
3. Посчитайте число объектов каждого класса и долю `Survived = 1`.
4. Найдите столбцы с особенно большим количеством пропусков.

Используйте компактные выражения pandas: `shape`, `dtypes`, `isna()`, `value_counts()`.

In [ ]:
# TODO: первичная диагностика датасета
# Ожидаемые результаты сохраните в переменных:
# missing_by_column
# target_counts
# target_share

### Интерпретация

Заполните ответы прямо в этой Markdown-ячейке.

1. Какие три столбца содержат пропуски?
2. Какой столбец имеет настолько много пропусков, что в обязательном baseline мы его исключим?
3. Сбалансированы ли классы идеально? Почему одной accuracy в такой задаче может быть недостаточно?

**Ваш ответ:**

- ...

## 2.2. Фиксируем признаки для обязательного baseline

В реальном проекте выбор признаков сам по себе был бы частью эксперимента. Здесь набор фиксируется,
чтобы в ЛР3 можно было менять **модель**, не меняя входные данные.

| Роль | Признаки |
|---|---|
| Числовые | `Age`, `SibSp`, `Parch`, `Fare` |
| Категориальные | `Pclass`, `Sex`, `Embarked` |
| Не используем в обязательном baseline | `PassengerId`, `Name`, `Ticket`, `Cabin` |
| Target | `Survived` |

`Pclass` хранится как число, но обозначает класс обслуживания, поэтому в этой работе мы трактуем его как категориальный признак.

> Исключение признака не означает, что он «бесполезен вообще». Это лишь фиксированное решение для общего учебного baseline.

In [ ]:
TARGET = "Survived"

NUMERIC_FEATURES = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_FEATURES = ["Pclass", "Sex", "Embarked"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURES].copy()
y = df[TARGET].copy()

assert X.shape == (891, 7)
assert y.shape == (891,)
assert set(y.unique()) == {0, 1}

display(X.head())

# 3. Минимальный EDA: график должен отвечать на вопрос

EDA здесь не превращается в отдельную лабораторную по визуализации.

Для начала посмотрим на простой табличный срез: долю выживших в зависимости от пола.

In [ ]:
survival_by_sex = (
    df.groupby("Sex")["Survived"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "survival_rate"})
)

display(survival_by_sex)

## 3.1. Задание 2 — три осмысленные визуализации

Постройте:

1. долю выживших для разных значений `Sex`;
2. долю выживших для разных `Pclass`;
3. распределение `Age` отдельно для `Survived = 0` и `Survived = 1`.

Для каждого графика должны быть заголовок и подписи осей.

После графиков сформулируйте **не больше 1–2 предложений на каждый**:
что видно и может ли соответствующий признак быть полезен модели.

In [ ]:
# TODO: три визуализации.
# Используйте отдельный plt.figure(...) для каждого графика.

### Интерпретация EDA

1. Что видно по `Sex`?
2. Что видно по `Pclass`?
3. Насколько очевидна связь `Age` с target только по этому графику?
4. Почему заметная зависимость на графике ещё не доказывает причинную связь?

**Ваш ответ:**

- ...

# 4. Фиксируем train / validation / test

Теперь начинается собственно эксперимент.

Используем:

- `train` — 70%;
- `validation` — 15%;
- `test` — 15%;
- `random_state = 42`;
- `stratify = y`.

**Validation** используется для анализа baseline и принятия решений.  
**Test** считаем отложенной финальной проверкой: до раздела 8 мы к его метрикам не обращаемся.

Важный принцип:

> Все статистики preprocessing — медианы, средние, параметры scaling, категории для encoding — должны оцениваться **только по train**.

## 4.1. Задание 3 — создать воспроизводимый split

Сначала отделите 15% test. Затем разделите оставшиеся 85% так, чтобы validation составлял 15% от всей выборки.

Сохраняйте именно **индексы строк**:

- `train_idx`
- `val_idx`
- `test_idx`

После этого создайте `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`.

Используйте `stratify` на обоих этапах.

In [ ]:
# TODO: создать train/validation/test split.
# Подсказка: доля validation внутри оставшихся 85% равна 0.15 / 0.85.

In [ ]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n": [len(y_train), len(y_val), len(y_test)],
    "share": [len(y_train) / len(y), len(y_val) / len(y), len(y_test) / len(y)],
    "survived_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
})

display(split_summary)

assert len(train_idx) + len(val_idx) + len(test_idx) == len(df)
assert set(train_idx).isdisjoint(val_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(val_idx).isdisjoint(test_idx)

### Почему здесь нужен `stratify`?

Ответьте до перехода дальше.

1. Что сохраняет `stratify=y`?
2. Особенно когда это важно?
3. Почему недостаточно один раз «удачно» получить похожие доли классов без фиксированного seed?

**Ваш ответ:**

- ...

# 5. Data leakage: маленькая утечка всё равно остаётся утечкой

Рассмотрим очень простой пример.

`Age` содержит пропуски. Если мы решили заполнять их медианой, возникает вопрос:

> **По каким данным считать медиану?**

Посчитаем два значения — пока только для демонстрации.

In [ ]:
full_age_median = df["Age"].median()
train_age_median = X_train["Age"].median()

print("Медиана Age по ВСЕМ данным:", full_age_median)
print("Медиана Age только по TRAIN:", train_age_median)

### Интерпретация leakage

1. Почему использовать `full_age_median` для заполнения пропусков было бы методически неправильно?
2. Делает ли небольшая разница между двумя медианами такую утечку допустимой?
3. Что изменится, если preprocessing будет оформлен как часть sklearn `Pipeline`?

**Ваш ответ:**

- ...

# 6. Preprocessing: разные признаки — разные преобразования

Наши признаки неоднородны.

Числовая ветка:

```text
Age, SibSp, Parch, Fare
        ↓
SimpleImputer(strategy="median")
        ↓
StandardScaler
```

Категориальная ветка:

```text
Pclass, Sex, Embarked
        ↓
SimpleImputer(strategy="most_frequent")
        ↓
OneHotEncoder(handle_unknown="ignore")
```

Обе ветки объединяются через `ColumnTransformer`.

`handle_unknown="ignore"` важен потому, что validation/test теоретически могут содержать категорию,
которой не было в train.

## 6.1. Задание 4 — собрать `ColumnTransformer`

Создайте функцию `make_preprocessor()`, которая каждый раз возвращает **новый** preprocessing pipeline.

Требования:

- числовые пропуски → median;
- числовые признаки → `StandardScaler`;
- категориальные пропуски → most frequent;
- категориальные признаки → `OneHotEncoder`;
- `sparse_output=False`;
- неизвестные категории не должны приводить к ошибке.

После этого создайте `preprocessor = make_preprocessor()`.

In [ ]:
# TODO: реализуйте make_preprocessor() и создайте preprocessor.

## 6.2. Fit только на train

Теперь важен не только **какой** метод вызвать, но и **на каких данных**.

Сделайте:

```python
X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared   = preprocessor.transform(X_val)
X_test_prepared  = preprocessor.transform(X_test)
```

Обратите внимание: `fit_transform` вызывается только для train.

In [ ]:
# TODO: обучите preprocessing на train и преобразуйте все три split.
# Также получите feature_names = preprocessor.get_feature_names_out()

In [ ]:
print("До preprocessing:", X_train.shape)
print("После preprocessing:", X_train_prepared.shape)
print("\nИтоговые признаки:")
print(feature_names)

assert X_train_prepared.shape[0] == len(y_train)
assert X_val_prepared.shape[0] == len(y_val)
assert X_test_prepared.shape[0] == len(y_test)

assert X_train_prepared.shape[1] == X_val_prepared.shape[1] == X_test_prepared.shape[1]
assert np.isfinite(X_train_prepared).all()
assert np.isfinite(X_val_prepared).all()
assert np.isfinite(X_test_prepared).all()

### Что произошло с размерностью?

Исходно было 7 признаков, после preprocessing их стало больше.

1. Почему?
2. Какой именно преобразователь увеличил число столбцов?
3. Почему `Pclass` дал несколько выходных признаков?
4. Что означает `num__` и `cat__` в `feature_names`?

**Ваш ответ:**

- ...

# 7. Baseline: сначала проверяем, лучше ли модель тривиального решения

Число `accuracy = 0.80` само по себе мало что говорит.

Сначала построим **DummyClassifier**, который всегда выбирает самый частый класс.
После этого сравним его с Logistic Regression.

Основные метрики:

- **accuracy** — доля правильных ответов;
- **F1** — баланс precision и recall для положительного класса;
- **confusion matrix** — какие именно ошибки делает модель.

In [ ]:
def get_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

## 7.1. Задание 5 — Dummy baseline

Обучите:

```python
DummyClassifier(strategy="most_frequent")
```

на `y_train`.

Поскольку DummyClassifier фактически игнорирует признаки, можно передать ему
`X_train_prepared` и затем получить `dummy_val_pred` для `X_val_prepared`.

Сохраните метрики в `dummy_val_metrics`.

In [ ]:
# TODO: DummyClassifier и его метрики на validation.

## 7.2. Задание 6 — Logistic Regression внутри полного Pipeline

Теперь соберите модель так, чтобы preprocessing и классификатор были единым объектом:

```text
raw DataFrame
    ↓
ColumnTransformer
    ↓
LogisticRegression
```

Создайте `logreg_pipeline`, обучите его **на raw `X_train`** и получите `logreg_val_pred` для raw `X_val`.

Используйте:

```python
LogisticRegression(max_iter=1000, random_state=SEED)
```

Сохраните метрики в `logreg_val_metrics`.

In [ ]:
# TODO: полный sklearn Pipeline + LogisticRegression.

In [ ]:
val_results = pd.DataFrame(
    [dummy_val_metrics, logreg_val_metrics],
    index=["DummyClassifier", "LogisticRegression"],
)

display(val_results.round(3))

ConfusionMatrixDisplay.from_predictions(y_val, logreg_val_pred)
plt.title("Logistic Regression — validation")
plt.tight_layout()
plt.show()

### Интерпретация validation-результата

1. Почему у DummyClassifier может быть не совсем маленькая accuracy, но F1 равен нулю?
2. Лучше ли Logistic Regression тривиального baseline? По каким признакам вы это решили?
3. Какой тип ошибок виден в confusion matrix?
4. Почему сейчас мы смотрим validation, а не test?

**Ваш ответ:**

- ...

# 8. Порог решения: одна вероятность — разные precision и recall

`LogisticRegression` умеет выдавать не только готовый класс, но и оценку вероятности положительного класса через `predict_proba`. Класс по умолчанию получается при пороге около `0.5`, но сам порог — часть правила принятия решения.

На **validation** сравните три порога: `0.3`, `0.5`, `0.7`. Для каждого посчитайте `precision` и `recall`. Затем постройте ROC-кривую и получите ROC-AUC по вероятностям. Test пока не используем.

> Смысл блока — увидеть компромисс: снижение порога обычно помогает находить больше положительных объектов (recall), но может увеличить число false positive и снизить precision.


## 8.1. Задание 7 — threshold, precision/recall и ROC-AUC

1. Получите `logreg_val_proba` — вероятность класса `1` для `X_val`.
2. Для порогов `0.3`, `0.5`, `0.7` сформируйте бинарные предсказания и соберите таблицу `threshold_metrics` со столбцами `threshold`, `precision`, `recall`.
3. Посчитайте `val_roc_auc = roc_auc_score(y_val, logreg_val_proba)`.
4. Постройте ROC-кривую.


In [ ]:
# TODO: исследуйте влияние threshold на precision/recall и посчитайте ROC-AUC.


### Интерпретация порога

1. Как изменились precision и recall при переходе от `0.3` к `0.7`?
2. Какой порог вы бы выбрали, если false negative особенно дороги? Почему?
3. Почему ROC-AUC считается по вероятностям и не привязан к одному выбранному threshold?
4. Почему выбирать threshold по test было бы нарушением роли test?

**Ваш ответ:**

- ...


# 9. Финальная проверка на test и несколько реальных ошибок

Pipeline зафиксирован. Только теперь используем test.

Мы не будем менять признаки или параметры Logistic Regression после просмотра test-метрик:
иначе test постепенно превратится ещё в один validation set.

## 9.1. Задание 8 — test metrics

Получите:

- `test_pred`;
- `test_metrics`.

Выведите accuracy, F1 и confusion matrix.

In [ ]:
# TODO: финальная оценка logreg_pipeline на X_test.

## 9.2. Посмотреть на конкретные ошибки

Aggregate-метрики полезны, но не показывают сами объекты.

Создайте `errors`, где будут исходные признаки test, правильный класс `y_true`,
предсказание `y_pred` и только ошибочно классифицированные строки.

Покажите минимум 5 ошибок.

In [ ]:
# TODO: создайте DataFrame errors и покажите несколько строк.

### Короткий анализ ошибок

Посмотрите на несколько false positive и false negative.

1. Есть ли среди ошибок группы, которые повторяются?
2. Можно ли по нескольким строкам уверенно объяснить причину каждой ошибки?
3. Почему анализ отдельных объектов дополняет confusion matrix, но не заменяет её?

**Ваш ответ:**

- ...

# 10. Передаём те же данные в ЛР3

В следующей лабораторной мы будем сравнивать MLP из sklearn и PyTorch.

Чтобы сравнение было честным, **данные не будем готовить заново**.
Используем fitted preprocessing из `logreg_pipeline` и сохраним:

- `X_train`, `X_val`, `X_test` после preprocessing;
- `y_train`, `y_val`, `y_test`;
- имена итоговых признаков;
- отдельную таблицу `PassengerId → split`.

Так в ЛР3 изменится модель, а не исходные условия эксперимента.

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

fitted_preprocessor = logreg_pipeline.named_steps["preprocessor"]

X_train_lr3 = np.asarray(
    fitted_preprocessor.transform(X_train), dtype=np.float32
)
X_val_lr3 = np.asarray(
    fitted_preprocessor.transform(X_val), dtype=np.float32
)
X_test_lr3 = np.asarray(
    fitted_preprocessor.transform(X_test), dtype=np.float32
)

feature_names_lr3 = np.asarray(
    fitted_preprocessor.get_feature_names_out(), dtype=str
)

np.savez_compressed(
    ARTIFACT_DIR / "lr2_titanic_prepared.npz",
    X_train=X_train_lr3,
    X_val=X_val_lr3,
    X_test=X_test_lr3,
    y_train=y_train.to_numpy(dtype=np.int64),
    y_val=y_val.to_numpy(dtype=np.int64),
    y_test=y_test.to_numpy(dtype=np.int64),
    feature_names=feature_names_lr3,
)

split_labels = np.empty(len(df), dtype=object)
split_labels[train_idx] = "train"
split_labels[val_idx] = "validation"
split_labels[test_idx] = "test"

split_table = pd.DataFrame({
    "PassengerId": df["PassengerId"],
    "split": split_labels,
})

split_table.to_csv(
    ARTIFACT_DIR / "lr2_titanic_split.csv",
    index=False,
)

print("Сохранено:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path)

In [ ]:
# Финальные sanity checks

assert X_train_lr3.dtype == np.float32
assert X_val_lr3.dtype == np.float32
assert X_test_lr3.dtype == np.float32

assert X_train_lr3.shape[1] == X_val_lr3.shape[1] == X_test_lr3.shape[1]
assert len(feature_names_lr3) == X_train_lr3.shape[1]

assert np.isfinite(X_train_lr3).all()
assert np.isfinite(X_val_lr3).all()
assert np.isfinite(X_test_lr3).all()

assert len(split_table) == len(df)
assert set(split_table["split"]) == {"train", "validation", "test"}

with np.load(ARTIFACT_DIR / "lr2_titanic_prepared.npz") as saved:
    assert saved["X_train"].shape == X_train_lr3.shape
    assert saved["y_train"].shape == y_train.shape

print("Финальные проверки пройдены.")

# 11. Вывод

Ответьте коротко, но содержательно — примерно 8–12 предложений суммарно.

1. **Ответ на центральный вопрос:** как подготовка данных влияет на корректность эксперимента?
2. Зачем split выполняется до вычисления статистик preprocessing?
3. Как `Pipeline` снижает риск leakage и делает эксперимент воспроизводимым?
4. Зачем нужен DummyClassifier, если мы всё равно хотим использовать более сильную модель?
5. Что показало изменение threshold о компромиссе precision/recall и что дополнительно сообщает ROC-AUC?
6. Что именно мы зафиксировали для честного сравнения моделей в ЛР3?

**Ваш вывод:**

...


# 12. Источники и использованные инструменты

### Материалы курса

- `titanic.csv` из исходных материалов курса;
- задания и пояснения лабораторной работы №2.

### Внешние источники

Перечислите только существенные материалы, которые реально использовали:

- ...

### ИИ-инструменты

Например: ChatGPT, GitHub Copilot и т. п.

- ...

Историю промптов прикладывать не требуется. На защите нужно понимать и уметь изменить ключевой код независимо от его происхождения.

# 13. Завершение практики A и подготовка к практике B

| Компонент | Баллы | Что проверяется в ЛР2 |
|---|---:|---|
| Корректность реализации | 3 | EDA, split, preprocessing, Pipeline, baseline, threshold/ROC-AUC и экспорт данных |
| Эксперимент и метрики | 2 | честный split, отсутствие leakage, Dummy vs Logistic Regression, метрики |
| Анализ и выводы | 1 | интерпретация EDA, confusion matrix, ошибок и центрального вопроса |
| Устная защита | 4 | понимание split, preprocessing, leakage, encoding, scaling и метрик |

**Hard gate:** менее **2 из 4** баллов за устную часть — лабораторная не засчитывается.

На защите будьте готовы:

- объяснить, почему конкретный столбец числовой/категориальный/исключённый;
- объяснить разницу между `fit_transform` и `transform`;
- показать, где именно preprocessing обучается только на train;
- объяснить `stratify`, validation и test;
- интерпретировать confusion matrix;
- выполнить небольшое live-изменение и заранее спрогнозировать его эффект.

---

### Самопроверка перед сдачей

- [ ] Все `TODO` заполнены.
- [ ] Notebook запускается сверху вниз без ошибок.
- [ ] Используется `SEED = 42`.
- [ ] Train / validation / test не пересекаются.
- [ ] Preprocessing обучается только на train.
- [ ] Test не использовался для выбора решений до финальной оценки.
- [ ] Есть 3 осмысленные визуализации и краткая интерпретация.
- [ ] DummyClassifier сравнивается с Logistic Regression.
- [ ] Посчитаны accuracy, F1 и confusion matrix.
- [ ] Просмотрены конкретные ошибки модели.
- [ ] Созданы `artifacts/lr2_titanic_prepared.npz` и `artifacts/lr2_titanic_split.csv`.
- [ ] Написан ответ на центральный вопрос.
- [ ] Указаны существенные источники и ИИ-инструменты.
- [ ] Вы готовы объяснить и изменить ключевой код на защите.